# Initialization

In [5]:
import json
import uuid
import os
import json
from dotenv import load_dotenv
from pathlib import Path
from kafka import KafkaProducer
from faker import Faker
from time import sleep
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, expr, to_timestamp
from pyspark.sql.types import StructType, StringType, DoubleType, TimestampType

# spark = (
#     SparkSession 
#     .builder 
#     .appName("Dibimbing Spark-Kafka") 
#     .config("spark.streaming.stopGracefullyOnShutdown", True) 
#     .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.2')
#     .config("spark.sql.shuffle.partitions", 4)
#     .master("local[*]") 
#     .getOrCreate()
# )

# spark

In [ ]:
# Load environment variables
dotenv_path = Path('/resources/.env')
load_dotenv(dotenv_path=dotenv_path)

kafka_host = os.getenv('KAFKA_HOST')
kafka_topic = os.getenv('KAFKA_TOPIC_NAME')

# Initialize Spark session
spark = (
    SparkSession
    .builder
    .appName("Dibimbing Spark-Kafka")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", 5)
    .config("spark.streaming.stopGracefullyOnShutdown", True)
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.2")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# Define the expected schema of JSON messages
schema = StructType() \
    .add("transaction_id", StringType()) \
    .add("customer_name", StringType()) \
    .add("category", StringType()) \
    .add("amount", DoubleType()) \
    .add("payment_type", StringType()) \
    .add("location", StringType()) \
    .add("timestamp", StringType())  # This is Unix time as a string

# Read from Kafka as streaming DataFrame
kafka_df = (
    spark
    .readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", f"{kafka_host}:9092")
    .option("subscribe", kafka_topic)
    .option("startingOffsets", "earliest")
    .load()
)

# Parse JSON and convert timestamp
parsed_df = (
    kafka_df
    .selectExpr("CAST(value AS STRING) as json_str")
    .select(from_json(col("json_str"), schema).alias("data"))
    .select("data.*")
    .withColumn("event_time", to_timestamp(col("timestamp").cast("long")))
)

# Preview schema
parsed_df.printSchema()

# Output stream to console
query = (
    parsed_df
    .writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", False)
    .start()
)

query.awaitTermination()

root
 |-- transaction_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- location: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- event_time: timestamp (nullable = true)

